# Entrenamiento del VAE multi-resolucion (`scripts/VAE.py`)

Entrena `MultiResVAE` sobre los stamps multi-banda / multi-resolucion de
COSMOS, usando el pipeline de preprocesamiento de `scripts/preprocess_vae.py`.
Pensado para correr en Google Colab con GPU, pero funciona igual en local.

Estructura:
1. Setup del entorno (Colab o local)
2. Datos y preprocesamiento -> `file_out_data/vae_input.npz`
3. Configuracion del entrenamiento (hiperparametros de `scripts/VAE.py`)
4. Entrenamiento con monitoreo en vivo (loss, KL, dimensiones activas)
5. Resultados: reconstrucciones, muestras del prior, espacio latente (UMAP)
6. Persistir checkpoint + figuras + metricas en Google Drive (el runtime de
   Colab es efimero: sin este paso se pierden al desconectar)


## 1. Setup del entorno (Colab o local)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Rama de trabajo actual (donde vive scripts/VAE.py con los ultimos cambios).
# Cambiar a "master" si ya hiciste merge.
REPO_BRANCH = "update-preprocess-vae"


def _run(cmd):
    """subprocess en vez de `!cmd`: devuelve returncode y siempre imprime
    stdout/stderr, para poder diagnosticar un clone fallido (red, rate-limit
    de GitHub, etc.) en vez de un error generico de 'no encontrado'."""
    print(f"$ {' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout)
    if proc.stderr.strip():
        print(proc.stderr)
    return proc.returncode


if IN_COLAB:
    REPO_URL = "https://github.com/Tamaracarrasco/AS4501-Proyect.git"
    repo_root = Path("/content/AS4501-Proyect")

    # Pararse en un directorio que seguro existe (y que no vamos a borrar)
    # ANTES de tocar repo_root: si una corrida anterior dejo el kernel con
    # cwd == repo_root y despues lo borramos, cualquier subprocess (git
    # incluido) falla con "Unable to read current working directory".
    os.chdir("/content")

    def _repo_ready(path):
        return (path / "scripts" / "VAE.py").exists()

    if repo_root.exists() and not _repo_ready(repo_root):
        print(f"{repo_root} existe pero no tiene scripts/VAE.py; se re-clona.")
        _run(["rm", "-rf", str(repo_root)])

    if not repo_root.exists():
        rc = _run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
                    "--single-branch", REPO_URL, str(repo_root)])
        if rc != 0:
            raise RuntimeError(
                f"git clone termino con codigo {rc}; revisa el output de arriba "
                "(suele ser un problema de red o rate-limit de GitHub en Colab: "
                "reintenta la celda)."
            )

    if not _repo_ready(repo_root):
        raise FileNotFoundError(
            f"{repo_root} no tiene scripts/VAE.py tras clonar la rama '{REPO_BRANCH}'. "
            "Revisa el output del clone arriba."
        )

    os.chdir(repo_root)

    # Dependencias extra que no vienen preinstaladas en Colab. No se instala
    # requirements.txt completo a proposito: fija torch==...+cpu y arrastraria
    # paquetes pesados innecesarios, pisando el torch+CUDA que Colab ya trae.
    !pip install -q umap-learn astropy

    from google.colab import drive
    drive.mount("/content/drive")
else:
    repo_root = Path.cwd().resolve()
    for candidate in [repo_root, repo_root.parent, repo_root.parent.parent]:
        if (candidate / "scripts" / "VAE.py").exists():
            repo_root = candidate
            break
    else:
        raise FileNotFoundError(
            "No se encontro scripts/VAE.py desde el directorio actual."
        )

sys.path.insert(0, str(repo_root / "scripts"))

import torch

print(f"IN_COLAB   : {IN_COLAB}")
print(f"repo_root  : {repo_root}")
print(f"CUDA       : {torch.cuda.is_available()}")
if IN_COLAB and not torch.cuda.is_available():
    print("[aviso] Sin GPU: Entorno de ejecucion > Cambiar tipo de entorno > GPU (T4).")


## 2. Datos y preprocesamiento (`scripts/preprocess_vae.py`)

`data/features_images_20260707.csv` viaja en el repo (se trae solo con el
`git clone`). El tar de stamps (`cosmos_TC_202602.tar.gz.part_aa`, ~2 GB) es
demasiado pesado para git: hay que dejarlo en Google Drive y apuntar
`COSMOS_TAR_DRIVE_PATH` hacia el, o copiarlo a mano dentro de `data/`.

Si `file_out_data/vae_input.npz` ya existe **se reutiliza tal cual**:
`preprocess_vae.py` es determinista (seed=42), asi que si vas a comparar esta
corrida con vae1/vae2/vae3 conviene partir del mismo split train/val/test.


In [ ]:
# Editar solo si data/cosmos_TC_202602.tar.gz.part_aa no esta disponible
# localmente y hay que leerlo directo desde Drive (streaming, sin copiarlo).
COSMOS_TAR_DRIVE_PATH = "/content/drive/MyDrive/AS4501/cosmos_TC_202602.tar.gz.part_aa"

npz_path = repo_root / "file_out_data" / "vae_input.npz"
local_tar = repo_root / "data" / "cosmos_TC_202602.tar.gz.part_aa"

if not npz_path.exists() and not local_tar.exists() and IN_COLAB and Path(COSMOS_TAR_DRIVE_PATH).exists():
    os.environ["COSMOS_TAR"] = COSMOS_TAR_DRIVE_PATH
    print(f"COSMOS_TAR -> {COSMOS_TAR_DRIVE_PATH}")


In [ ]:
from preprocess_vae import main as preprocess_main

tar_path = Path(os.environ.get("COSMOS_TAR", local_tar))
features_path = repo_root / "data" / "features_images_20260707.csv"

if npz_path.exists():
    print(f"Ya existe {npz_path}, se omite el preprocesamiento.")
else:
    if not tar_path.exists():
        raise FileNotFoundError(
            f"No se encontro el tar de stamps en {tar_path}.\n"
            "Monta Drive y define COSMOS_TAR_DRIVE_PATH (celda anterior) o "
            "copia el tar a data/ antes de correr esta celda."
        )
    if not features_path.exists():
        raise FileNotFoundError(f"No se encontro el CSV de features: {features_path}")
    preprocess_main(tar_path=str(tar_path), features_path=str(features_path))


## 3. Configuracion del entrenamiento

`scripts/VAE.py` centraliza todos los hiperparametros en un dict `CONFIG`.
Partimos de una copia de ese dict (no mutar el original importado) y
sobreescribimos lo que haga falta. El resto de las claves de `CONFIG`
(`lr_enc`, `lr_dec`, `free_bits`, `viz_level`, `share_level_weights`, etc. —
ver `scripts/VAE.py`) quedan en sus valores por defecto salvo que las
agregues aca.


In [ ]:
from VAE import CONFIG as VAE_CONFIG, main as train_vae

config = dict(VAE_CONFIG)
config["data_path"] = str(npz_path)
config["out_dir"] = str(repo_root / "file_out_data")
config["run_id"] = None  # None = autoincrementa (no pisa vae1/vae2/vae3 existentes)

# Hiperparametros mas comunes de ajustar:
config["z_dim"] = 64
config["epochs"] = 200
config["batch_size"] = 128
config["beta_final"] = 1.0
config["warmup_epochs"] = 30
config["patience"] = 25
config["fig_every"] = 50          # cada cuantas epocas guarda recon/prior/umap
config["metrics_every"] = 10      # cada cuantas epocas calcula SSIM/PSNR (caras)
config["data_augmentation"] = False
config["use_wandb"] = False       # True (+ wandb logueado) manda metricas a wandb.ai

for k in ["z_dim", "epochs", "batch_size", "beta_final", "warmup_epochs", "patience"]:
    print(f"{k:18s}: {config[k]}")


### Monitoreo en vivo

`main()` acepta un `epoch_callback` opcional que se llama al final de cada
epoca; lo usamos para refrescar un grafico de loss/KL/dimensiones activas
directamente en el notebook mientras entrena, sin esperar al final.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output

PLOT_EVERY = 1  # refresca el grafico cada N epocas (subir si el notebook se pone lento)


def live_plot_callback(epoch, history, model, data, device, cfg):
    if epoch % PLOT_EVERY != 0:
        return
    clear_output(wait=True)
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs, [r["loss/recon_train"] for r in history], label="train")
    axes[0].plot(epochs, [r["loss/recon_val"] for r in history], label="val", ls="--")
    axes[0].set_title("recon"); axes[0].set_yscale("log"); axes[0].legend()

    axes[1].plot(epochs, [r["loss/kl_train"] for r in history], label="train")
    axes[1].plot(epochs, [r["loss/kl_val"] for r in history], label="val", ls="--")
    axes[1].set_title("KL"); axes[1].set_yscale("log"); axes[1].legend()

    axes[2].plot(epochs, [r["val/active_dims"] for r in history], color="steelblue")
    axes[2].set_title(f"dims activas (de {cfg['z_dim']})")
    axes[2].set_xlabel("epoch")

    fig.suptitle(f"vae{cfg['run_id']} - epoca {epoch}/{cfg['epochs']}")
    plt.tight_layout()
    plt.show()

    last = history[-1]
    print(f"e{epoch:03d} | beta={last['beta']:.3f} | "
          f"train recon={last['loss/recon_train']:.1f} kl={last['loss/kl_train']:.1f} | "
          f"val recon={last['loss/recon_val']:.1f} kl={last['loss/kl_val']:.1f} | "
          f"active={last['val/active_dims']}/{cfg['z_dim']}")


## 4. Entrenamiento

`run_id` se auto-incrementa mirando los `vae_summary*.json` existentes en
`file_out_data/`, asi que no pisa las corridas vae1/vae2/vae3 ya guardadas en
el repo. Con early stopping (`patience`), puede terminar antes de llegar a
`epochs`.


In [ ]:
model = train_vae(config, epoch_callback=live_plot_callback)


## 5. Resultados

In [ ]:
import json

run_id = config["run_id"]
fig_dir = Path(config["out_dir"]) / "figures" / f"vae{run_id}"
summary_path = Path(config["out_dir"]) / f"vae_summary{run_id}.json"

print(json.dumps(json.loads(summary_path.read_text()), indent=2))

loss_curves = fig_dir / "loss_curves.png"
if loss_curves.exists():
    plt.figure(figsize=(9, 9))
    plt.imshow(plt.imread(loss_curves))
    plt.axis("off")
    plt.title(loss_curves.name)
    plt.show()

# recon / prior / umap de la mejor epoca (el nombre de archivo incluye la epoca)
for pattern in ["recon_e*.png", "prior_e*.png", "umap_e*.png"]:
    matches = sorted(fig_dir.glob(pattern))
    if not matches:
        continue
    p = matches[-1]
    plt.figure(figsize=(14, 5))
    plt.imshow(plt.imread(p))
    plt.axis("off")
    plt.title(p.name)
    plt.show()


## 6. Persistir resultados en Google Drive

El almacenamiento de Colab (`/content`) es efimero: si el runtime se
desconecta se pierde el checkpoint entrenado. Copia las salidas a Drive antes
de cerrar la sesion (en local esto no hace falta: ya quedan en `file_out_data/`
del repo).


In [ ]:
DRIVE_OUT_DIR = "/content/drive/MyDrive/AS4501/file_out_data"  # editar si aplica

if IN_COLAB:
    dst = Path(DRIVE_OUT_DIR)
    dst.mkdir(parents=True, exist_ok=True)
    out_dir = Path(config["out_dir"])

    _run(["cp", "-v", str(out_dir / f"vae_summary{run_id}.json"), str(dst)])
    _run(["cp", "-v", str(out_dir / f"vae_metrics{run_id}.csv"), str(dst)])
    _run(["cp", "-v", str(out_dir / f"vae{run_id}.input"), str(dst)])
    _run(["cp", "-v", str(out_dir / "checkpoints" / f"vae_best{run_id}.pt"), str(dst)])
    _run(["cp", "-r", str(fig_dir), str(dst / f"figures_vae{run_id}")])
    print(f"Copiado a {dst}")
else:
    print("No estas en Colab; las salidas ya quedaron en el repo local (file_out_data/).")


## Notas

- `config['run_id']=None` autoincrementa mirando `vae_summary*.json`
  existentes: no pisa las corridas ya guardadas (vae1, vae2, vae3).
- Para barrer varios hiperparametros de una, `scripts/VAE.py` soporta
  `run_sweep(path, base_config)`, leyendo un archivo de texto
  (`# z_dim beta ...` de header + una fila por corrida). No se arma ese
  archivo aca porque no hay barrido definido todavia; ver la seccion
  "Barrido" al final de `scripts/VAE.py` si hace falta.
- `config['use_wandb']=True` (con `wandb` instalado y logueado) manda las
  metricas en vivo a https://wandb.ai ademas de los CSV/PNG locales.
- El checkpoint entrenado (`vae_best{run_id}.pt`) queda en el mismo formato
  que `vae_best2.pt` / `vae_best3.pt`, asi que es compatible directo con
  `scripts/Regression_XGBoost_Latent.ipynb` (basta con agregarlo a
  `MODEL_RUNS` ahi).
